### Ecuación de Advección
$$ \frac{\partial u}{\partial t} + v \cdot \nabla u = 0 \quad \text{en }\Omega$$
Formulación variacional, usando elementos discontinuos
$$\int_\Omega \left( \frac{\partial u}{\partial t} + v \cdot \nabla u\right)w + \int_E \left(|v\cdot n| - \frac12 v\cdot n\right)[u] w = 0$$

In [ ]:
%reset -f
import mfem.ser as mfem
import numpy as np
from scipy.special import erfc

#### Clase para el problema de evolución

In [ ]:
class FE_Evolution(mfem.PyTimeDependentOperator):
    def __init__(self, M, K, ge=0):
        mfem.PyTimeDependentOperator.__init__(self, M.Size())

        self.K = K
        self.M = M
        self.T = None
        
        self.z = mfem.Vector(M.Size())
        
        self.M_prec = mfem.DSmoother()
        self.M_solver = mfem.CGSolver()
        self.M_solver.SetPreconditioner(self.M_prec)
        self.M_solver.SetOperator(M)
        self.M_solver.iterative_mode = False
        self.M_solver.SetRelTol(1e-9)
        self.M_solver.SetAbsTol(0.0)
        self.M_solver.SetMaxIter(100)
        self.M_solver.SetPrintLevel(0)

        self.T_prec = mfem.BlockILU(ge)
        self.T_solver = mfem.GMRESSolver()
        self.T_solver.iterative_mode = False
        self.T_solver.SetRelTol(1.e-8)
        self.T_solver.SetAbsTol(0.0)
        self.T_solver.SetMaxIter(500)
        self.T_solver.SetPrintLevel(0)
        self.T_solver.SetPreconditioner(self.T_prec)       

    def Mult(self, x, y):
        self.K.Mult(x, self.z)
        self.z.Neg()
        self.M_solver.Mult(self.z, y)

    def ImplicitSolve(self, dt, x, y):
        if self.T is None:
            self.T = mfem.Add(1.0, self.M, dt, self.K)
            self.T_solver.SetOperator(self.T)
        
        self.K.Mult(x, self.z)
        self.z.Neg()
        self.T_solver.Mult(self.z, y)       
        

#### Malla

In [ ]:
ref_levels = 3
meshfile = 'mallas/periodic-hexagon.mesh'

mesh = mfem.Mesh(meshfile)
dim = mesh.Dimension()

for lev in range(ref_levels):
    mesh.UniformRefinement()

#### Selección de solver para la edo

In [ ]:
ode_solver_type = 2

if ode_solver_type == 1:
    ode_solver = mfem.ForwardEulerSolver()
elif ode_solver_type == 2:
    ode_solver = mfem.RK2Solver(1.0)
elif ode_solver_type == 3:
    ode_solver = mfem.RK3SSolver()
elif ode_solver_type == 4:
    ode_solver = mfem.RK4Solver()
elif ode_solver_type == 6:
    ode_solver = mfem.RK6Solver()
elif ode_solver_type == 11:
    ode_solver = mfem.BackwardEulerSolver()

#### Espacio de elementos finitos discontinuo
La opción `BasisType.GaussLobato` usa puntos de cuadratura de Gauss-Lobato, a diferencia de la opción por defecto que usas `BasisType.GaussLegendre`.

In [ ]:
order = 3
fec = mfem.L2_FECollection(order, dim, mfem.BasisType.GaussLobatto)
fes = mfem.FiniteElementSpace(mesh, fec)

#### Velocidad y dato inicial

In [ ]:
    
bb_min, bb_max = mesh.GetBoundingBox()
center = (bb_min + bb_max)/2.0
        
class velocity_coeff(mfem.VectorPyCoefficient):
    def EvalValue(self, x):
        X = 2 * (x - center) / (bb_max - bb_min)
        v = [np.pi/2*X[1],  - np.pi/2*X[0]]
        return v

class u0_coeff(mfem.PyCoefficient):
    def EvalValue(self, x):
        X = 2 * (x - center) / (bb_max - bb_min)
        rx = 0.45
        ry = 0.25
        cx = 0.
        cy = -0.2
        w = 10.
        return (erfc(w * (X[0]-rx)) * erfc(-w*(X[0]+rx)) *
                erfc(w * (X[1]-cy-ry)) * erfc(-w*(X[1]-cy+ry)))/16.

velocity = velocity_coeff(dim)
u0 = u0_coeff()

#### Formulación variacional

In [ ]:
m = mfem.BilinearForm(fes)
m.AddDomainIntegrator(mfem.MassIntegrator())
k = mfem.BilinearForm(fes)
k.AddDomainIntegrator(mfem.ConvectionIntegrator(velocity, 1.0))
k.AddInteriorFaceIntegrator(
    mfem.NonconservativeDGTraceIntegrator(velocity, 1.0))
k.AddBdrFaceIntegrator(
        mfem.NonconservativeDGTraceIntegrator(velocity, 1.0))

m.Assemble()
m.Finalize()
k.Assemble()
k.Finalize()

#### Condición inicial

In [ ]:
u = mfem.GridFunction(fes)
u.ProjectCoefficient(u0)

In [ ]:
t_final = 5.
dt = 0.001
vis_step = 100

sol_sock = mfem.socketstream("localhost", 19916)
sol_sock.precision(8)
sol_sock.send_solution(mesh,  u)

adv = FE_Evolution(m.SpMat(), k.SpMat(), fes.GetTypicalFE().GetDof())
ode_solver.Init(adv)

t = 0.0
ti = 0

while True:
    if t > t_final - dt/2:
        break
    t, dt = ode_solver.Step(u, t, dt)
    ti +=1

    if ti % vis_step == 0:
        cad = "Time: {0:2.2f}".format(t)
        print("time step:",ti,"time:",np.round(t, 3))
        sol_sock.send_solution(mesh,  u)
        sol_sock.send_text("plot_caption '{0:s}'".format(cad))